In [0]:
%pip install torch torchvision tqdm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Import Library

In [0]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, models, transforms
import mlflow
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
import base64
import io
from PIL import Image
from tqdm import tqdm
import os

Konfigurasi Path dan Parameter

In [0]:
IMAGE_PATH = "/Volumes/workspace/churn_analytics/image_store"

# Parameter
NUM_EPOCHS = 5
BATCH_SIZE = 4
LEARNING_RATE = 0.001

Persiapan Data

In [0]:
# Transformasi gambar: resize, ubah jadi tensor, normalisasi
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("Loading image data from Volume...")
# 'ImageFolder' secara otomatis menggunakan nama folder (ktp, tagihan) sebagai label
image_dataset = datasets.ImageFolder(IMAGE_PATH, data_transforms)
dataloader = DataLoader(image_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Simpan nama-nama kelas
class_names = image_dataset.classes
num_classes = len(class_names)
print(f"Kelas ditemukan: {class_names}")

Loading image data from Volume...
Kelas ditemukan: ['ktp', 'tagihan']


PERSIAPAN MODEL (TRANSFER LEARNING RESNET18)

In [0]:
model = models.resnet18(pretrained=True)

# Bekukan semua layer kecuali layer terakhir
for param in model.parameters():
    param.requires_grad = False

# Ganti layer terakhir (fc) agar sesuai dengan jumlah kelas kita (num_classes)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

# Pindahkan model ke GPU jika ada, jika tidak CPU
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.fc.parameters(), lr=LEARNING_RATE, momentum=0.9)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-c93abcfe-cc39-4b31-b01f-86e8d5b1bf71/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-c93abcfe-cc39-4b31-b01f-86e8d5b1bf71/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/spark-c93abcfe-cc39-4b31-b01f-86/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 284MB/s]


LOOP TRAINING MODEL

In [0]:
for epoch in range(NUM_EPOCHS):
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
    model.train()
    running_loss = 0.0

    for inputs, labels in tqdm(dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(image_dataset)
    print(f'Loss: {epoch_loss:.4f}')

Epoch 1/5


100%|██████████| 3/3 [00:02<00:00,  1.13it/s]


Loss: 0.8765
Epoch 2/5


100%|██████████| 3/3 [00:01<00:00,  2.54it/s]


Loss: 0.6821
Epoch 3/5


...

Loss: 0.6395
Epoch 4/5


100%|██████████| 3/3 [00:01<00:00,  2.67it/s]


Loss: 0.4474
Epoch 5/5


100%|██████████| 3/3 [00:01<00:00,  2.51it/s]

Loss: 0.3789


WRAPPER MODEL UNTUK DEPLOYMENT (BASE64 -> PREDIKSI)

In [0]:
# Wrapper ini penting agar Postman bisa mengirim gambar sebagai teks Base64
class ImageModelWrapper(mlflow.pyfunc.PythonModel):
    
    def load_context(self, context):
        # Load nama kelas dari artefak (file teks)
        with open(context.artifacts["class_names"], 'r') as f:
            self.class_names = f.read().split(',')
        num_classes = len(self.class_names)

        self.model = models.resnet18(weights=None) 
        for param in self.model.parameters():
            param.requires_grad = False
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)
        
        # Load *state_dict* (weights) ke dalam struktur model
        self.model.load_state_dict(torch.load(context.artifacts["model_path"]))
        
        # Atur device dan mode .eval()
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval() 

        # Definisikan ulang transformasi
        self.transforms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def predict(self, context, model_input):
        def process_image(img_b64):
            img_bytes = base64.b64decode(img_b64)
            img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
            tensor = self.transforms(img).unsqueeze(0).to(self.device)
            
            with torch.no_grad():
                output = self.model(tensor)
                _, pred_index = torch.max(output, 1)
                return self.class_names[pred_index.item()]
        
        return model_input["image_base64"].apply(process_image)


/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


LOGGING MODEL KE MLFLOW

In [0]:
# Buat path sementara untuk menyimpan artefak
model_path = "resnet_doc_classifier.pth"
class_names_path = "class_names.txt"

torch.save(model.state_dict(), model_path)

with open(class_names_path, 'w') as f:
    f.write(",".join(class_names))

# Input: 1 kolom string "image_base64"
input_schema = Schema([ColSpec("string", "image_base64")])
output_schema = Schema([ColSpec("string")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

with mlflow.start_run(run_name="Model Klasifikasi Dokumen") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model_wrapper", # Nama folder artefak
        python_model=ImageModelWrapper(),
        artifacts={
            "model_path": model_path,
            "class_names": class_names_path
        },
        signature=signature,
    )

    mlflow.log_param("num_classes", num_classes)
    mlflow.log_param("epochs", NUM_EPOCHS)

print(f"Model berhasil di-log dengan Run ID: {run.info.run_id}")

# Hapus file sementara
os.remove(model_path)
os.remove(class_names_path)

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/__init__.py:3224: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Model berhasil di-log dengan Run ID: 1e68ec692f5045ec8e3ae3a3048e763e
